# UHDM + DCID 快速路线实验
需要 GPU、Internet、原图 DCID 子集、UHDM Kaggle 数据，以及旧 fusion_training_results.zip。只需运行本 Notebook，不运行旧训练。默认两个分支各 300 步。

In [ ]:
%pip install -q gdown pyyaml scikit-image


In [ ]:
import types
H=types.ModuleType("baseline_helpers")
exec('"""复制整个文件到 Kaggle 新单元格执行。先安装 gdown pyyaml scikit-image。\n独立入口：不依赖原 UHDM 数据集、不加载融合模型、不训练。\n"""\nfrom pathlib import Path\nimport csv, io, json, hashlib, zipfile, sys, subprocess, importlib.util, time\nfrom datetime import datetime\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps, ImageDraw\nimport torch\nimport torch.nn.functional as F\nimport yaml\nfrom skimage.metrics import structural_similarity\nfrom tqdm.auto import tqdm\nfrom IPython.display import display, FileLink\nDATA_SOURCE = None\nLOCAL_WEIGHT = None\nREGION = 768\nSCALES = (1.0, 0.5)\nMAX_PAIRS = 20\nROOT = Path(\'/kaggle/working/dcid_baseline\')\n\ndef sha_file(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\nclass PairSource:\n\n    def __init__(self, source):\n        self.path = Path(source)\n        self.archive = zipfile.ZipFile(self.path) if self.path.suffix.lower() == \'.zip\' else None\n        if self.archive:\n            members = [n for n in self.archive.namelist() if n.endswith(\'_manifest_draft.csv\')]\n            if len(members) != 1:\n                raise RuntimeError(f\'ZIP 内应有唯一原始清单，实际为 {members}\')\n            self.manifest_bytes = self.archive.read(members[0])\n            self.prefix = members[0].rsplit(\'/\', 1)[0] + \'/\' if \'/\' in members[0] else \'\'\n        else:\n            self.manifest_bytes = self.path.read_bytes()\n        self.rows = list(csv.DictReader(io.StringIO(self.manifest_bytes.decode(\'utf-8-sig\'), newline=\'\')))\n        if not self.rows:\n            raise RuntimeError(\'清单为空\')\n        if not {\'sample_id\', \'input\', \'target\'} <= set(self.rows[0]):\n            raise RuntimeError(\'清单缺少必要字段\')\n        ids = [r[\'sample_id\'] for r in self.rows]\n        if len(ids) != len(set(ids)):\n            raise RuntimeError(\'sample_id 重复，请检查清单\')\n\n    def read_bytes(self, row, key):\n        name = row[key].replace(\'\\\\\', \'/\')\n        relative = \'images/\' + name.split(\'/images/\', 1)[1] if \'/images/\' in name else name\n        if self.archive:\n            return self.archive.read(self.prefix + relative)\n        p = Path(name)\n        if not p.is_file():\n            p = self.path.parent / relative\n        if not p.is_file():\n            raise FileNotFoundError(f\'{key}: {name}，也未找到 {p}\')\n        return p.read_bytes()\n\n    def read(self, row, key):\n        with Image.open(io.BytesIO(self.read_bytes(row, key))) as image:\n            return np.array(image.convert(\'RGB\'), copy=True)\n\ndef locate_source():\n    if DATA_SOURCE is not None:\n        return Path(DATA_SOURCE)\n    preferred = Path(\'/kaggle/working/dcid_hf_subset/train_iphone_1_part1_20_2026_manifest_draft.csv\')\n    if preferred.is_file():\n        return preferred\n    for pattern in [\'*_manifest_draft.csv\', \'*_with_images.zip\']:\n        found = sorted((p for base in [Path(\'/kaggle/working\'), Path(\'/kaggle/input\')] if base.exists() for p in base.rglob(pattern)))\n        if len(found) == 1:\n            return found[0]\n        if len(found) > 1:\n            raise RuntimeError(f\'候选不唯一，请设置 DATA_SOURCE：{found}\')\n    raise FileNotFoundError(\'没有找到 DCID 清单或 ZIP。请恢复下载结果，或将原图子集挂载为 Kaggle Input。\')\n\ndef load_official_model():\n    repo = Path(\'/kaggle/working/moire_project/UHDM\')\n    if not (repo / \'model/nets.py\').is_file():\n        repo = ROOT / \'UHDM\'\n        if not repo.exists():\n            subprocess.check_call([\'git\', \'clone\', \'--depth\', \'1\', \'https://github.com/CVMI-Lab/UHDM.git\', str(repo)])\n    commit = subprocess.check_output([\'git\', \'-C\', str(repo), \'rev-parse\', \'HEAD\'], text=True).strip()\n    if LOCAL_WEIGHT is not None:\n        weight = Path(LOCAL_WEIGHT)\n        if not weight.is_file():\n            raise FileNotFoundError(weight)\n    else:\n        known = [Path(\'/kaggle/working/moire_project/uhdm_checkpoint.pth\'), ROOT / \'uhdm_checkpoint.pth\']\n        weight = next((p for p in known if p.is_file()), None)\n        if weight is None and Path(\'/kaggle/input\').exists():\n            found = list(Path(\'/kaggle/input\').rglob(\'uhdm_checkpoint.pth\'))\n            if len(found) > 1:\n                raise RuntimeError(f\'找到多个权重，请指定 LOCAL_WEIGHT：{found}\')\n            if found:\n                weight = found[0]\n        if weight is None:\n            import gdown\n            weight = ROOT / \'uhdm_checkpoint.pth\'\n            temporary = weight.with_suffix(\'.download\')\n            gdown.download(id=\'1HT_ubcAYRqzFIJ46XuPhrulJk2YFBIEl\', output=str(temporary), quiet=False)\n            if not temporary.is_file():\n                raise RuntimeError(\'官方权重下载失败，请挂载旧权重并填写 LOCAL_WEIGHT\')\n            temporary.replace(weight)\n    spec = importlib.util.spec_from_file_location(\'dcid_esdnet\', repo / \'model/nets.py\')\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    cfg = yaml.safe_load((repo / \'config/uhdm_config.yaml\').read_text())[\'MODEL\']\n    model = module.my_model(en_feature_num=cfg[\'EN_FEATURE_NUM\'], en_inter_num=cfg[\'EN_INTER_NUM\'], de_feature_num=cfg[\'DE_FEATURE_NUM\'], de_inter_num=cfg[\'DE_INTER_NUM\'], sam_number=cfg[\'SAM_NUMBER\'])\n    state = torch.load(weight, map_location=\'cpu\', weights_only=True)\n    if \'state_dict\' in state:\n        state = state[\'state_dict\']\n    model.load_state_dict({k.removeprefix(\'module.\'): v for k, v in state.items()}, strict=True)\n    return (model.cuda().eval().requires_grad_(False), {\'repo_commit\': commit, \'weight_sha256\': sha_file(weight), \'weight_path\': str(weight)})\n\n@torch.inference_mode()\ndef predict(model, array, scale):\n    h, w = array.shape[:2]\n    x = torch.from_numpy(array.copy()).permute(2, 0, 1)[None].float().cuda() / 255\n    sh, sw = (round(h * scale), round(w * scale))\n    if (sh, sw) != (h, w):\n        x = F.interpolate(x, size=(sh, sw), mode=\'bilinear\', align_corners=False, antialias=True)\n    x = F.pad(x, (0, -sw % 32, 0, -sh % 32), mode=\'replicate\')\n    y = model(x)\n    y = y[0] if isinstance(y, (tuple, list)) else y\n    y = y[:, :, :sh, :sw]\n    if (sh, sw) != (h, w):\n        y = F.interpolate(y, size=(h, w), mode=\'bilinear\', align_corners=False)\n    return y.clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()\n\ndef metrics(pred, gt):\n    pred = np.clip(pred, 0, 1).astype(np.float32)\n    gt = gt.astype(np.float32)\n    mse = float(np.mean((pred.astype(np.float64) - gt.astype(np.float64)) ** 2))\n    return {\'psnr\': float(-10 * np.log10(max(mse, 1e-12))), \'ssim\': float(structural_similarity(gt, pred, data_range=1.0, channel_axis=-1, gaussian_weights=True, sigma=1.5, use_sample_covariance=False))}\n\ndef preview(images, path):\n    sheet = Image.new(\'RGB\', (320 * len(images), 350), \'white\')\n    draw = ImageDraw.Draw(sheet)\n    for i, (label, array) in enumerate(images):\n        im = Image.fromarray(np.round(np.clip(array, 0, 1) * 255).astype(\'uint8\'))\n        sheet.paste(ImageOps.contain(im, (315, 315)), (i * 320, 30))\n        draw.text((i * 320 + 5, 5), label, fill=\'black\')\n    sheet.save(path, quality=95)\n\ndef main():\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'当前没有 GPU。先保存下载结果；GPU 空闲后，在 Notebook 设置中启用 GPU 再运行。\')\n    ROOT.mkdir(parents=True, exist_ok=True)\n    source = PairSource(locate_source())\n    rows = source.rows[:MAX_PAIRS]\n    fingerprints = []\n    for row in tqdm(rows, desc=\'检查配对\'):\n        a, g = (source.read(row, \'input\'), source.read(row, \'target\'))\n        if a.shape != g.shape or min(a.shape[:2]) < REGION:\n            raise RuntimeError(f\'配对尺寸不适用：{row[\'sample_id\']} {a.shape} {g.shape}\')\n        fingerprints.append({\'sample_id\': row[\'sample_id\'], **{k + \'_sha256\': hashlib.sha256(source.read_bytes(row, k)).hexdigest() for k in [\'input\', \'target\']}})\n    out = ROOT / datetime.now().strftime(\'center768_%Y%m%d_%H%M%S_%f\')\n    out.mkdir()\n    model, metadata = load_official_model()\n    protocol = {**metadata, \'data_source\': str(source.path), \'manifest_sha256\': hashlib.sha256(source.manifest_bytes).hexdigest(), \'fingerprints\': fingerprints, \'region\': REGION, \'crop\': \'native-resolution center; identical for all methods\', \'scales\': SCALES, \'pairs\': len(rows), \'torch\': torch.__version__, \'gpu\': torch.cuda.get_device_name(0), \'training\': False, \'status\': \'exploratory target training-pool subset; not official test or full-image score\', \'ssim\': \'RGB, gaussian sigma1.5, population covariance, data_range1\', \'timing\': \'synchronized predict call incl CPU/GPU transfer; no speedup claim; first image includes warmup overhead\'}\n    (out / \'protocol.json\').write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    pd.DataFrame(rows).to_csv(out / \'evaluated_manifest.csv\', index=False)\n    results = []\n    for index, row in enumerate(tqdm(rows, desc=\'DCID 基线推理\')):\n        a, g = (source.read(row, \'input\'), source.read(row, \'target\'))\n        h, w = a.shape[:2]\n        top, left = ((h - REGION) // 2, (w - REGION) // 2)\n        a = a[top:top + REGION, left:left + REGION].copy()\n        gt = g[top:top + REGION, left:left + REGION].astype(np.float32) / 255\n        x = a.astype(np.float32) / 255\n        images = [(\'Input\', x)]\n        folder = out / f\'{index:02d}\'\n        folder.mkdir()\n        Image.fromarray(a).save(folder / \'input.png\')\n        Image.fromarray(g[top:top + REGION, left:left + REGION]).save(folder / \'gt.png\')\n        base = {\'sample_id\': row[\'sample_id\'], \'index\': index, \'top\': top, \'left\': left, \'region\': REGION}\n        results.append({**base, \'method\': \'input\', **metrics(x, gt), \'seconds\': 0.0})\n        for scale in SCALES:\n            torch.cuda.synchronize()\n            start = time.perf_counter()\n            try:\n                pred = predict(model, a, scale)\n            except torch.cuda.OutOfMemoryError:\n                torch.cuda.empty_cache()\n                raise RuntimeError(\'显存不足：当前实验停止，不混合不同区域的成绩。将 REGION 改成 512 后整轮重跑；结果目录会独立保存。\') from None\n            torch.cuda.synchronize()\n            seconds = time.perf_counter() - start\n            method = f\'esd_scale_{scale:g}\'\n            results.append({**base, \'method\': method, **metrics(pred, gt), \'seconds\': seconds})\n            Image.fromarray(np.round(pred * 255).astype(\'uint8\')).save(folder / f\'{method}.png\')\n            images.append((method, pred))\n        preview(images + [(\'GT\', gt)], folder / \'comparison.jpg\')\n        pd.DataFrame(results).to_csv(out / \'per_image.csv\', index=False)\n    frame = pd.DataFrame(results)\n    summary = frame.groupby(\'method\').agg(psnr=(\'psnr\', \'mean\'), ssim=(\'ssim\', \'mean\'), images=(\'index\', \'nunique\'))\n    summary.to_csv(out / \'summary.csv\')\n    pivot = frame.pivot(index=\'sample_id\', columns=\'method\', values=\'psnr\')\n    delta = pivot.sub(pivot[\'input\'], axis=0).drop(columns=\'input\')\n    delta.to_csv(out / \'gain_vs_input.csv\')\n    display(summary)\n    print(\'相对输入的逐图 PSNR 增益统计：\')\n    display(delta.agg([\'mean\', \'median\', \'min\', \'max\']))\n    print(\'正增益图片数：\')\n    print((delta > 0).sum())\n    with Image.open(out / \'00/comparison.jpg\') as im:\n        display(im.copy())\n    bundle = out.parent / (out.name + \'_results.zip\')\n    with zipfile.ZipFile(bundle, \'w\', zipfile.ZIP_DEFLATED) as z:\n        for p in out.rglob(\'*\'):\n            if p.is_file():\n                z.write(p, p.relative_to(out))\n    print(\'完成。当前仅为中心区域开发诊断；不训练、不使用正式测试集、不宣称实际加速。\')\n    print(\'下载并上传此 ZIP：\', bundle)\n    display(FileLink(str(bundle)))\n    return out', H.__dict__)
from pathlib import Path
import io,csv,json,random,hashlib,zipfile,time,copy
from datetime import datetime
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from IPython.display import display,FileLink

# 只需调整本区；通常可保持默认。
DCID_SOURCE=None             # CSV 或原图 ZIP；自动查找不唯一时填写 Kaggle 路径
FUSION_ZIP=None              # 旧 fusion_training_results.zip 的 Kaggle 路径
WEIGHT=None                  # 原始 uhdm_checkpoint.pth，可自动下载
UHDM_ROOT=None               # 自动定位 soumikrakshit/uhdm-dataset；不下载整个数据集
STEPS=300                    # 两个分支各 300 步；先不加长
EVAL_EVERY=100
DCID_TRAIN=12                # 剩余 8 张作为开发验证
SOURCE_REPLAY=20             # 旧 100 张训练池中的前 20 张，保持旧 10 张验证不变
REGION=768
CROP=256
BATCH=2
LR=1e-5
SEED=2026
RUN_ROOT=Path('/kaggle/working/joint_moire_runs')

exec("def fusion_lowpass(x):\n    v = torch.arange(-4, 5, device=x.device, dtype=x.dtype)\n    k = torch.exp(-v.square() / 8)\n    k = k / k.sum()\n    c = x.shape[1]\n    a = F.conv2d(F.pad(x, (4, 4, 0, 0), mode='replicate'), k.view(1, 1, 1, 9).repeat(c, 1, 1, 1), groups=c)\n    return F.conv2d(F.pad(a, (0, 0, 4, 4), mode='replicate'), k.view(1, 1, 9, 1).repeat(c, 1, 1, 1), groups=c)\n\nclass ScaleFusion(nn.Module):\n\n    def __init__(self, mode='band', preferred=0):\n        super().__init__()\n        assert mode in ('spatial', 'band')\n        self.mode = mode\n        outputs = 6 if mode == 'band' else 3\n        self.net = nn.Sequential(nn.Conv2d(12, 24, 3, padding=1), nn.ReLU(), nn.Conv2d(24, 24, 3, padding=1), nn.ReLU(), nn.Conv2d(24, outputs, 1))\n        nn.init.zeros_(self.net[-1].weight)\n        nn.init.zeros_(self.net[-1].bias)\n        with torch.no_grad():\n            self.net[-1].bias[preferred] = 6.0\n            if mode == 'band':\n                self.net[-1].bias[preferred + 3] = 6.0\n\n    def forward(self, x, candidates, lows):\n        b, _, _, h, w = candidates.shape\n        features = torch.cat((x, candidates.reshape(b, 9, h, w)), dim=1)\n        features = F.interpolate(features, size=(max(1, h // 4), max(1, w // 4)), mode='area')\n        logits = F.interpolate(self.net(features), size=(h, w), mode='bilinear', align_corners=False)\n        wl = logits[:, :3].softmax(dim=1).unsqueeze(2)\n        wh = logits[:, 3:].softmax(dim=1).unsqueeze(2) if self.mode == 'band' else wl\n        return (wl * lows + wh * (candidates - lows)).sum(dim=1)",globals())

def find_unique(pattern,explicit):
    if explicit is not None:
        p=Path(explicit)
        if not p.is_file():raise FileNotFoundError(p)
        return p
    found=sorted(p for root in [Path('/kaggle/input'),Path('/kaggle/working')] if root.exists() for p in root.rglob(pattern))
    if len(found)!=1:raise RuntimeError(f'请在配置中填写 {pattern} 的路径，候选={found}')
    return found[0]

def load_old_fusion():
    # Kaggle 上传 ZIP 后可能自动展开，两种形态均支持。
    if FUSION_ZIP is not None and Path(FUSION_ZIP).is_dir():
        folder=Path(FUSION_ZIP)
    else:
        found=([Path(FUSION_ZIP)] if FUSION_ZIP is not None else
            [p for root in [Path('/kaggle/input'),Path('/kaggle/working')] if root.exists() for p in root.rglob('fusion_training_results.zip')])
        if len(found)>1:raise RuntimeError(f'旧融合包不唯一，请填写 FUSION_ZIP：{found}')
        if found:
            with zipfile.ZipFile(found[0]) as z:
                p=z.read('protocol.json');ck=z.read('band/best.pt')
            return json.loads(p),torch.load(io.BytesIO(ck),map_location='cpu',weights_only=True),hashlib.sha256(p+ck).hexdigest()
        folders=sorted({p.parent.parent for root in [Path('/kaggle/input'),Path('/kaggle/working')] if root.exists()
            for p in root.rglob('best.pt') if p.parent.name=='band' and (p.parent.parent/'protocol.json').is_file()})
        if len(folders)!=1:raise RuntimeError(f'请挂载旧融合结果包，FUSION_ZIP 可填写 ZIP 路径或展开后的根目录。候选={folders}')
        folder=folders[0]
    p=(folder/'protocol.json').read_bytes();ck=(folder/'band/best.pt').read_bytes()
    return json.loads(p),torch.load(io.BytesIO(ck),map_location='cpu',weights_only=True),hashlib.sha256(p+ck).hexdigest()

def source_path(old,root):
    p=Path(old)
    if p.is_file():return p
    tail=old.replace('\\','/').split('/uhdm-dataset/',1)[-1]
    direct=root/tail
    if direct.is_file():return direct
    # 兼容 train/train 包装差异：按 pair_xx/文件名匹配，不能只按文件名。
    matches=[p for p in root.rglob(Path(tail).name) if p.parent.name==Path(tail).parent.name]
    if len(matches)!=1:raise RuntimeError(f'UHDM 图片无法唯一映射：{old}; {matches}')
    return matches[0]

def crop_pair(a,g,label):
    if a.shape!=g.shape or min(a.shape[:2])<REGION:raise RuntimeError(f'尺寸异常：{label}')
    h,w=a.shape[:2];t,l=(h-REGION)//2,(w-REGION)//2
    return a[t:t+REGION,l:l+REGION].copy(),g[t:t+REGION,l:l+REGION].copy()

def load_source(items,root,split):
    output=[]
    for row in tqdm(items,desc='UHDM '+split):
        xp,gp=source_path(row['x'],root),source_path(row['gt'],root)
        with Image.open(xp) as im:a=np.array(im.convert('RGB'))
        with Image.open(gp) as im:g=np.array(im.convert('RGB'))
        a,g=crop_pair(a,g,str(gp))
        output.append({'id':'UHDM/'+gp.parent.name+'/'+gp.stem,'x':a,'gt':g,
            'gt_hash':H.sha_file(gp),'input_hash':H.sha_file(xp),
            'path':str(gp),'split':split,'group':str(row['group'])})
    return output

def tensor(a):return torch.from_numpy(a.copy()).permute(2,0,1)[None].float().cuda()/255

def anchor(y,x):return y+(x-y).mean(dim=(-2,-1),keepdim=True)

@torch.inference_mode()
def forward_native(model,a):
    x=tensor(a);h,w=x.shape[-2:]
    y=model(F.pad(x,(0,(-w)%32,0,(-h)%32),mode='replicate'))[0][:,:,:h,:w]
    return y,x

def arr(t):return t[0].detach().clamp(0,1).permute(1,2,0).cpu().numpy()

@torch.inference_mode()
def evaluate(model,items,label,domain,out,step,anchored=False,include_baselines=False,fusion=None):
    model.eval();records=[]
    for i,p in enumerate(items):
        raw,x=forward_native(model,p['x']);g=p['gt'].astype(np.float32)/255
        predictions={label:arr(anchor(raw,x) if anchored else raw)}
        if include_baselines:
            predictions={'input':p['x'].astype(np.float32)/255,'original':arr(raw),'mean_anchor':arr(anchor(raw.clamp(0,1),x))}
            if fusion is not None:
                candidates=torch.stack([torch.from_numpy(H.predict(model,p['x'],s)).permute(2,0,1).cuda() for s in (.5,.75,1.)])[None]
                lows=fusion_lowpass(candidates[0])[None]
                predictions['old_band']=arr(fusion(x,candidates,lows))
        for method,pred in predictions.items():
            score=H.metrics(pred,g)
            records.append({'domain':domain,'id':p['id'],'method':method,'step':step,**score})
            # 每张图保存，挑失败案例不需要重新推理。
            dest=out/'images'/domain/method;dest.mkdir(parents=True,exist_ok=True)
            Image.fromarray(np.round(pred*255).astype('uint8')).save(dest/f'{i:02d}.png')
        if i<3:
            dest=out/'previews';dest.mkdir(exist_ok=True)
            H.preview([('Input',p['x']/255)]+list(predictions.items())+[('GT',g)],dest/f'{domain}_{label}_{step}_{i}.jpg')
    return records

def check_split(training,validation):
    for field in ['gt_hash','input_hash']:
        overlap={p[field] for p in training}&{p[field] for p in validation}
        if overlap:raise RuntimeError(f'训练/验证有精确重复 {field}；请修复划分，尚未启动训练')

def draw_batch(items,rng):
    xs=[];gs=[]
    for _ in range(BATCH):
        p=rng.choice(items);t=rng.randrange(REGION-CROP+1);l=rng.randrange(REGION-CROP+1)
        a=p['x'][t:t+CROP,l:l+CROP];g=p['gt'][t:t+CROP,l:l+CROP]
        if rng.random()<.5:a,g=np.flip(a,1),np.flip(g,1)
        xs.append(tensor(a));gs.append(tensor(g))
    return torch.cat(xs),torch.cat(gs)

def save_bundle(out):
    bundle=out.parent/(out.name+'_results.zip')
    with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
        for p in out.rglob('*'):
            if p.is_file():z.write(p,p.relative_to(out))
    print('结果包：',bundle);display(FileLink(str(bundle)))

def run_joint():
    assert torch.cuda.is_available(),'需要 GPU；CPU 会话先保存数据即可'
    assert REGION%32==0 and CROP%32==0 and CROP<=REGION
    # 先检查挂载项，避免跑到训练中途才发现缺少 UHDM。
    protocol,checkpoint,old_fusion_hash=load_old_fusion()
    if UHDM_ROOT is None:
        roots=sorted(p for p in Path('/kaggle/input').rglob('uhdm-dataset') if p.is_dir())
        if len(roots)!=1:raise RuntimeError('请 Add Input 挂载 soumikrakshit/uhdm-dataset，或填写 UHDM_ROOT')
        source_root=roots[0]
    else:source_root=Path(UHDM_ROOT)
    H.DATA_SOURCE=DCID_SOURCE;source=H.PairSource(H.locate_source())
    rows=sorted(source.rows,key=lambda r:r['sample_id']);random.Random(SEED).shuffle(rows)
    assert len(rows)==20,'首轮固定使用已检查的 20 对，不自动加入其他图片'
    target=[]
    for i,row in enumerate(tqdm(rows,desc='DCID 配对')):
        a,g=crop_pair(source.read(row,'input'),source.read(row,'target'),row['sample_id'])
        target.append({'id':'DCID/'+row['sample_id'],'x':a,'gt':g,
            'gt_hash':hashlib.sha256(source.read_bytes(row,'target')).hexdigest(),
            'input_hash':hashlib.sha256(source.read_bytes(row,'input')).hexdigest(),
            'split':'train' if i<DCID_TRAIN else 'development_val','path':row['target']})
    target_train,target_val=target[:DCID_TRAIN],target[DCID_TRAIN:]
    source_train=load_source(protocol['manifest']['train'][:SOURCE_REPLAY],source_root,'replay_train')
    source_val=load_source(protocol['manifest']['val'],source_root,'old_development_val')
    check_split(source_train+target_train,source_val+target_val)
    if {p['group'] for p in source_train}&{p['group'] for p in source_val}:raise RuntimeError('UHDM 旧组划分有交集')
    out=RUN_ROOT/datetime.now().strftime('joint_%Y%m%d_%H%M%S');out.mkdir(parents=True)
    H.ROOT=RUN_ROOT/'official';H.ROOT.mkdir(exist_ok=True);H.LOCAL_WEIGHT=WEIGHT
    model,metadata=H.load_official_model()
    # 旧频段融合只用来评价，不重新训练，也不接在已适配的主干后面。
    digest=hashlib.sha256()
    for k,v in sorted(model.state_dict().items()):digest.update(k.encode());digest.update(v.detach().cpu().contiguous().numpy().tobytes())
    if digest.hexdigest()!=protocol['weight_sha256']:raise RuntimeError('原始主干与旧融合实验权重不一致，停止，不能混接')
    band=ScaleFusion('band',int(checkpoint['settings']['preferred'])).cuda().eval()
    band.load_state_dict(checkpoint['model'],strict=True)
    for p in band.parameters():p.requires_grad_(False)
    manifest=[{k:v for k,v in p.items() if k not in ['x','gt']} for p in target+source_train+source_val]
    pd.DataFrame(manifest).to_csv(out/'manifest.csv',index=False)
    settings={**metadata,'seed':SEED,'region':REGION,'crop':CROP,'batch':BATCH,'steps_per_branch':STEPS,
        'lr':LR,'source_replay_images':len(source_train),'source_val_images':len(source_val),
        'target_train_images':len(target_train),'target_val_images':len(target_val),
        'replay_schedule':'three DCID batches then one UHDM batch; total optimizer steps equal',
        'trainable':'decoder only; encoder frozen eval; MSE on unclipped input-mean-anchored prediction; adaptation eval anchors raw output then clips; fixed mean_anchor baseline anchors clipped original prediction',
        'old_fusion_protocol_and_band_sha256':old_fusion_hash,'old_protocol':protocol,
        'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0),
        'limitations':'DCID image split, scene grouping unknown, prior inspected data; UHDM old development val; no independent test claim. Fixed last step is primary; best target checkpoint is exploratory. Same total steps means different target update counts.'}
    (out/'protocol.json').write_text(json.dumps(settings,ensure_ascii=False,indent=2),encoding='utf-8')
    initial={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    records=[];history=[]
    try:
        print('阶段 1：两数据集固定基线与旧融合，先得到可下载结果')
        for domain,items in [('DCID',target_val),('UHDM',source_val)]:
            records+=evaluate(model,items,'baseline',domain,out,0,include_baselines=True,fusion=band)
        del band;torch.cuda.empty_cache()
        def persist():
            frame=pd.DataFrame(records);frame.to_csv(out/'per_image.csv',index=False)
            table=frame.groupby(['domain','method','step']).agg(psnr=('psnr','mean'),ssim=('ssim','mean'),images=('id','nunique'))
            table.to_csv(out/'summary.csv');pd.DataFrame(history).to_csv(out/'history.csv',index=False)
            return table
        display(persist());save_bundle(out)
        for branch in ['target_only','target_with_replay']:
            model.load_state_dict(initial,strict=True)
            model.requires_grad_(False);model.decoder.requires_grad_(True)
            opt=torch.optim.Adam(model.decoder.parameters(),lr=LR)
            rng=random.Random(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
            best=-float('inf');begin=time.perf_counter()
            for step in tqdm(range(1,STEPS+1),desc=branch):
                model.train();model.encoder.eval()
                pool=source_train if branch=='target_with_replay' and step%4==0 else target_train
                x,g=draw_batch(pool,rng)
                opt.zero_grad(set_to_none=True)
                y=model(x)[0];loss=F.mse_loss(anchor(y,x),g)
                if not torch.isfinite(loss):raise RuntimeError('loss 非有限值，停止')
                loss.backward();torch.nn.utils.clip_grad_norm_(model.decoder.parameters(),1.);opt.step()
                if step==20:
                    torch.cuda.synchronize();elapsed=time.perf_counter()-begin
                    print(f'{branch} 前20步 {elapsed:.1f}s；仅训练部分预计剩余 {(STEPS-step)*elapsed/20/60:.1f} 分钟，另有评价开销')
                if step%EVAL_EVERY==0 or step==STEPS:
                    metrics=[]
                    for domain,items in [('DCID',target_val),('UHDM',source_val)]:
                        metrics+=evaluate(model,items,branch,domain,out,step,anchored=True)
                    records+=metrics
                    score=float(np.mean([r['psnr'] for r in metrics if r['domain']=='DCID']))
                    history.append({'branch':branch,'step':step,'train_loss':float(loss.detach()),'target_val_psnr':score,
                                    'elapsed_seconds':time.perf_counter()-begin})
                    payload={'decoder':model.decoder.state_dict(),'step':step,'branch':branch,'settings':{k:v for k,v in settings.items() if k!='old_protocol'}}
                    torch.save(payload,out/(branch+'_last.pt'))
                    if score>best:
                        best=score;torch.save(payload,out/(branch+'_best_target_dev.pt'))
                    display(persist())
            del opt;save_bundle(out)
        print('完成：优先比较两个分支固定第300步与 mean_anchor；同时查看 UHDM 是否退化。')
        return out
    except Exception as error:
        (out/'error.txt').write_text(repr(error),encoding='utf-8')
        if records:pd.DataFrame(records).to_csv(out/'partial_per_image.csv',index=False)
        save_bundle(out)
        raise

if __name__=='__main__':
    JOINT_RUN=run_joint()
